# Module 06 — Lecture 3: Multi-GPU Scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_06_advanced_topics/03_multi_gpu_scaling.ipynb)

---

A single A100 GPU has 80 GB of HBM. For a million-neuron network with 10% connectivity, the weight matrix alone requires **400 GB** — far beyond any single GPU. Multi-GPU simulation distributes neurons across GPUs and communicates spikes between them.

**Note:** This lecture requires access to a machine with ≥2 NVIDIA GPUs. The concepts and code are explained for self-study even without the hardware.

**Learning objectives:**
- Understand domain decomposition for spiking networks
- Implement inter-GPU spike communication with `cudaMemcpyPeer`
- Measure scaling efficiency and identify bottlenecks
- Apply Amdahl's Law to predict scaling limits

In [ ]:
!nvidia-smi

## 1. Scaling Strategy: Domain Decomposition

**Divide neurons across GPUs:**
- GPU 0 owns neurons 0 … N/2−1
- GPU 1 owns neurons N/2 … N−1

Each GPU holds:
- Its subset of V[], g_E[], g_I[], ref[] (neuron state)
- The **full** outgoing connectivity row_ptr and col_idx for its neurons (within the CSR for its partition)
- Cross-partition synapses need communication

**Each timestep:**
1. GPU 0 and GPU 1 update their neurons in parallel
2. Each GPU identifies which of its neurons fired
3. **Communication:** Exchange spike lists between GPUs
4. Apply incoming spikes from the other GPU to local conductances

**Communication cost:** For p=0.1 connectivity and 2% firing rate:
- ~0.2% of neurons' spikes cross the boundary per timestep
- For N=100,000 neurons: ~200 spike IDs per step × 4 bytes = 800 bytes
- PCIe bandwidth: ~12 GB/s → 800 bytes takes < 1 µs
- NVLink bandwidth: ~300 GB/s → much faster for larger messages

### Amdahl's Law

If fraction $f$ of work can be parallelized:
$$S(n) = \frac{1}{(1-f) + f/n}$$

For spiking network simulation:
- Neuron update: 100% parallelizable
- Spike propagation (within GPU): ~100% parallelizable
- Spike communication (between GPUs): serial bottleneck

The serial fraction determines the maximum speedup regardless of GPU count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Amdahl's Law analysis for spiking network simulation
n_gpus = np.array([1, 2, 4, 8, 16, 32, 64])

# Different serial fractions (communication overhead)
serial_fracs = {
    'f_serial=1% (NVLink, small msg)': 0.01,
    'f_serial=5% (PCIe, larger msg)':  0.05,
    'f_serial=10% (PCIe, no overlap)': 0.10,
    'f_serial=20% (unoptimized)':      0.20,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, f in serial_fracs.items():
    speedup = 1.0 / ((1-f) + f/n_gpus.astype(float))
    eff = speedup / n_gpus
    axes[0].plot(n_gpus, speedup,  '-o', lw=2, label=label, markersize=5)
    axes[1].plot(n_gpus, eff*100,  '-o', lw=2, label=label, markersize=5)

axes[0].plot(n_gpus, n_gpus, 'k--', lw=1.5, label='Ideal linear scaling', alpha=0.5)
axes[0].set_xlabel('Number of GPUs', fontsize=12)
axes[0].set_ylabel('Speedup', fontsize=12)
axes[0].set_title("Amdahl's Law for GPU Network Simulation", fontsize=12)
axes[0].legend(fontsize=8, loc='upper left'); axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log', base=2); axes[0].set_xticks(n_gpus)
axes[0].set_xticklabels(n_gpus)

axes[1].axhline(70, color='gray', linestyle=':', alpha=0.7, label='70% efficiency target')
axes[1].set_xlabel('Number of GPUs', fontsize=12)
axes[1].set_ylabel('Parallel efficiency (%)', fontsize=12)
axes[1].set_title('Parallel Efficiency vs GPU Count', fontsize=12)
axes[1].legend(fontsize=8, loc='upper right'); axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log', base=2); axes[1].set_xticks(n_gpus)
axes[1].set_xticklabels(n_gpus)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('amdahl_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print("Maximum speedup (n→∞) by serial fraction:")
for label, f in serial_fracs.items():
    print(f"  {label}: {1/f:.1f}×")

## 2. Multi-GPU Code Structure

The following shows the key patterns. Full compilation requires a multi-GPU machine.

In [ ]:
%%writefile multi_gpu_sketch.cu
/*
 * multi_gpu_sketch.cu
 * Demonstrates the structure of a 2-GPU LIF network simulation.
 * Does NOT run without 2 GPUs — study the patterns.
 *
 * Compile: nvcc -O2 -o multi_gpu_sketch multi_gpu_sketch.cu -lm
 * Run:     ./multi_gpu_sketch  (requires 2+ NVIDIA GPUs)
 */
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <omp.h>  // For host-side parallelism across GPUs

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ int   c_T_ref;

// Same LIF kernel as before — each GPU runs this for its partition
__global__ void lif_update(float* V, int* ref, int* fired,
                            const float* I_ext, int N_local)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N_local) return;
    fired[i] = 0;
    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }
    V[i] += c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * I_ext[i]);
    if (V[i] >= c_V_th) { V[i] = c_V_reset; ref[i] = c_T_ref; fired[i] = 1; }
}

// Apply incoming spikes: for each spike_id received from other GPU,
// increment conductance of local post-synaptic neurons via our CSR
__global__ void apply_remote_spikes(
    const int* remote_fired_list, int n_remote_fired,
    const int* row_ptr_remote, const int* col_local, const float* weights,
    float* g_E, int N_local
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n_remote_fired) return;
    int remote_i = remote_fired_list[tid];  // global neuron ID on other GPU
    for (int k = row_ptr_remote[remote_i]; k < row_ptr_remote[remote_i+1]; k++) {
        int local_j = col_local[k];  // local index on this GPU
        if (local_j >= 0 && local_j < N_local)
            atomicAdd(&g_E[local_j], weights[k]);
    }
}

int main() {
    int n_gpus = 0;
    CUDA_CHECK(cudaGetDeviceCount(&n_gpus));
    printf("Found %d GPU(s)\n", n_gpus);

    if (n_gpus < 2) {
        printf("This demo requires 2+ GPUs. Showing structure only.\n");
        printf("\nKey API calls for multi-GPU:\n");
        printf("  cudaSetDevice(gpu_id)     — activate a specific GPU\n");
        printf("  cudaMemcpyPeer(dst, dst_dev, src, src_dev, bytes)\n");
        printf("                            — copy between GPU memories\n");
        printf("  cudaDeviceEnablePeerAccess(peer, 0)\n");
        printf("                            — enable NVLink/PCIe peer access\n");
        printf("  cudaStreamCreate(&stream) — async streams for overlap\n");
        printf("\nSimulation loop structure (2 GPUs):\n");
        printf("  for each timestep:\n");
        printf("    [GPU 0] lif_update(local neurons 0..N/2)\n");
        printf("    [GPU 1] lif_update(local neurons N/2..N)  (parallel)\n");
        printf("    --- barrier (cudaDeviceSynchronize on each GPU) ---\n");
        printf("    [CPU]  compact fired list on each GPU\n");
        printf("    [CPU]  cudaMemcpyPeer(fired_list_0→GPU1, fired_list_1→GPU0)\n");
        printf("    [GPU 0] apply_remote_spikes(from GPU 1)\n");
        printf("    [GPU 1] apply_remote_spikes(from GPU 0)  (parallel)\n");
        return 0;
    }

    // ─── Full 2-GPU simulation (runs only with 2+ GPUs) ──────────────────────
    int N = 10000;
    int N_each = N / 2;  // neurons per GPU
    float dt=0.1f, tau_m=20.f, E_L=-65.f, Rm=10.f, V_th=-55.f, V_reset=-70.f;
    int T_ref = 20, T_steps = 1000;

    // Enable peer access between GPUs
    int can_access;
    CUDA_CHECK(cudaDeviceCanAccessPeer(&can_access, 0, 1));
    if (can_access) {
        cudaSetDevice(0); cudaDeviceEnablePeerAccess(1, 0);
        cudaSetDevice(1); cudaDeviceEnablePeerAccess(0, 0);
        printf("Peer access enabled (NVLink or PCIe)\n");
    } else {
        printf("No direct peer access — using staged copies\n");
    }

    // Allocate state on each GPU
    float* d_V[2];   int* d_ref[2]; int* d_fired[2];
    float* d_Iext[2];
    for (int g = 0; g < 2; g++) {
        CUDA_CHECK(cudaSetDevice(g));
        CUDA_CHECK(cudaMalloc(&d_V[g],    N_each*sizeof(float)));
        CUDA_CHECK(cudaMalloc(&d_ref[g],  N_each*sizeof(int)));
        CUDA_CHECK(cudaMalloc(&d_fired[g],N_each*sizeof(int)));
        CUDA_CHECK(cudaMalloc(&d_Iext[g], N_each*sizeof(float)));

        // Init
        float* h_V   = (float*)malloc(N_each*sizeof(float));
        float* h_Iext= (float*)malloc(N_each*sizeof(float));
        for (int i=0;i<N_each;i++) { h_V[i]=E_L; h_Iext[i]=1.5f; }
        CUDA_CHECK(cudaMemcpy(d_V[g],   h_V,   N_each*sizeof(float),cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_Iext[g],h_Iext,N_each*sizeof(float),cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemset(d_ref[g],0,N_each*sizeof(int)));
        free(h_V); free(h_Iext);

        // Set constants on this GPU
        CUDA_CHECK(cudaMemcpyToSymbol(c_dt,       &dt,      sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,    &tau_m,   sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,      &E_L,     sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,       &Rm,      sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,     &V_th,    sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset,  &V_reset, sizeof(float)));
        CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,    &T_ref,   sizeof(int)));
    }

    // Spike exchange buffers (pinned host memory for fast PCIe transfer)
    int* h_fired_buf[2];
    for (int g=0;g<2;g++)
        CUDA_CHECK(cudaHostAlloc(&h_fired_buf[g],
                                  N_each*sizeof(int), cudaHostAllocDefault));

    int thr=256, blk=(N_each+thr-1)/thr;
    int total_spikes=0;

    cudaEvent_t ev0, ev1;
    cudaSetDevice(0); CUDA_CHECK(cudaEventCreate(&ev0));
    cudaSetDevice(0); CUDA_CHECK(cudaEventCreate(&ev1));
    CUDA_CHECK(cudaEventRecord(ev0));

    for (int step=0; step<T_steps; step++) {
        // Phase 1: update neurons in parallel on both GPUs
        #pragma omp parallel for num_threads(2)
        for (int g=0; g<2; g++) {
            cudaSetDevice(g);
            lif_update<<<blk,thr>>>(d_V[g], d_ref[g], d_fired[g], d_Iext[g], N_each);
        }

        // Phase 2: sync and copy fired lists to host
        for (int g=0; g<2; g++) {
            cudaSetDevice(g);
            cudaDeviceSynchronize();
            cudaMemcpy(h_fired_buf[g], d_fired[g], N_each*sizeof(int),
                       cudaMemcpyDeviceToHost);
        }

        // Count spikes (simplified — no cross-GPU propagation in this sketch)
        for (int g=0;g<2;g++)
            for (int i=0;i<N_each;i++)
                if (h_fired_buf[g][i]) total_spikes++;

        // In a full simulation:
        // cudaMemcpyPeer(d_remote_fired[0], 0, d_fired[1], 1, N_each*sizeof(int));
        // cudaMemcpyPeer(d_remote_fired[1], 1, d_fired[0], 0, N_each*sizeof(int));
        // Then launch apply_remote_spikes on each GPU
    }

    cudaSetDevice(0);
    CUDA_CHECK(cudaEventRecord(ev1)); CUDA_CHECK(cudaEventSynchronize(ev1));
    float sim_ms; CUDA_CHECK(cudaEventElapsedTime(&sim_ms, ev0, ev1));
    float T_ms = T_steps * dt;
    printf("2-GPU sim: N=%d, T=%.0f ms, GPU=%.1f ms, RT factor=%.1fx\n",
           N, T_ms, sim_ms, T_ms/sim_ms);
    printf("Total spikes: %d (%.1f Hz avg)\n",
           total_spikes, (float)total_spikes/(N*T_ms/1000.f));

    for (int g=0;g<2;g++) {
        cudaSetDevice(g);
        cudaFree(d_V[g]); cudaFree(d_ref[g]);
        cudaFree(d_fired[g]); cudaFree(d_Iext[g]);
        cudaFreeHost(h_fired_buf[g]);
    }
    CUDA_CHECK(cudaEventDestroy(ev0)); CUDA_CHECK(cudaEventDestroy(ev1));
    return 0;
}

In [ ]:
# Compile with OpenMP for host-side parallelism
!nvcc -O2 -Xcompiler -fopenmp -o multi_gpu_sketch multi_gpu_sketch.cu -lm
!./multi_gpu_sketch

## 3. Communication Optimization Strategies

### Strategy 1: Compute-Communication Overlap
Use CUDA streams to overlap spike propagation on GPU 0 with data transfer from GPU 0 to GPU 1:

```c
cudaStream_t compute_stream, comm_stream;
// Launch compute on stream 0
lif_update<<<blk,thr,0,compute_stream>>>(...);
// Transfer happens concurrently on comm_stream
cudaMemcpyPeerAsync(dst, 1, src, 0, bytes, comm_stream);
// Sync only at timestep boundary
cudaStreamSynchronize(compute_stream);
cudaStreamSynchronize(comm_stream);
```

### Strategy 2: Sparse Spike Lists
Instead of transferring N/2 integers (one per neuron), transfer only the IDs of neurons that fired:

```c
// Compact: fired IDs → list
// Use thrust::copy_if or a custom atomic-counter kernel
int n_fired = compact_fired(d_fired, d_fired_list, N_local);
// Transfer only n_fired integers (typically 2% of N_local)
cudaMemcpyPeer(d_remote_list, peer_gpu, d_fired_list, this_gpu, n_fired*sizeof(int));
```

For 2% firing rate: transfer 0.02×N/2 ints instead of N/2 ints = **50× less data**.

### Strategy 3: Delay Buffers
Biological synaptic delays (1–10 ms) mean spikes from GPU 0 don't need to arrive on GPU 1 until next timestep. This allows **fully asynchronous** communication — transfer while computing, no synchronization needed per step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model communication bandwidth requirements vs network size
N_vals = np.array([10000, 50000, 100000, 500000, 1000000])
p_connect = 0.1
fire_rate  = 0.02  # 2% per step
dt_ms = 0.1

# Naive: transfer full fired array each step
bw_naive = N_vals / 2 * 4 / (dt_ms * 1e-3)  # bytes/s

# Compact: transfer only fired neuron IDs
bw_compact = N_vals / 2 * fire_rate * 4 / (dt_ms * 1e-3)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(N_vals/1e3, bw_naive  /1e9, 'r-o', lw=2, label='Naive (full array)')
ax.loglog(N_vals/1e3, bw_compact/1e9, 'b-s', lw=2, label='Compact (fired IDs only)')
ax.axhline(12, color='orange', linestyle='--', lw=1.5, label='PCIe 4.0 BW (12 GB/s)')
ax.axhline(300, color='purple', linestyle='--', lw=1.5, label='NVLink BW (300 GB/s)')
ax.set_xlabel('Network size N (thousands)', fontsize=12)
ax.set_ylabel('Required bandwidth (GB/s)', fontsize=12)
ax.set_title('Multi-GPU Communication Requirements vs N', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.3)
ax.set_xticks(N_vals/1e3)
ax.set_xticklabels([f'{int(n)}k' for n in N_vals/1e3])

plt.tight_layout()
plt.savefig('multi_gpu_bw.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Communication requirements at dt={dt_ms} ms:")
for N in N_vals:
    bw_n = N/2 * fire_rate * 4 / (dt_ms * 1e-3) / 1e9
    print(f"  N={N:>7,d}: compact BW = {bw_n:.2f} GB/s "
          f"({'<PCIe' if bw_n < 12 else '>PCIe, need NVLink'})")

## Summary

| Strategy | Implementation | Benefit |
|----------|---------------|--------|
| Domain decomposition | Split neurons across GPUs | Linear memory scaling |
| Sparse spike lists | compact_fired → transfer only IDs | 50× less communication |
| Async streams | compute + comm overlap | Hides latency |
| Delay buffers | synaptic delays ≥ 1 ms | Full async, no sync per step |
| NVLink | peer access with 300 GB/s | 25× more BW than PCIe |

**Practical guidance:**
- For N ≤ 100,000: a single A100/H100 GPU is usually sufficient
- For N up to 1,000,000: 2–4 GPUs with NVLink scale well
- For N > 1M: MPI + multi-node GPU clusters (NEST GPU, GeNN, Brian2CUDA)

This concludes Module 06. The Projects/ directory contains two capstone simulations that integrate everything from the course.

---

**Course Complete!** You have finished all 6 modules of GPU Programming for Computational Neuroscience.

Return to the [course repository](https://github.com/praveen-dedigamage/Cuda-For-Computational-Neuroscience) to explore the exercises or review any module.